# Week 1 · Day 5 — Multi-step Brochure Generator

So far we've made *single* LLM calls. This notebook chains several together into
a small pipeline that turns a company's website into a marketing brochure:

1. **Scrape** the landing page and collect every link on it.
2. **Ask the model** which of those links matter for a brochure (About, Careers,
   Products — not Terms, Privacy, or social icons).
3. **Gather** the page text plus those relevant links.
4. **Ask the model again** to write the brochure from everything we collected.

The key idea: the LLM is used both as a **filter** (step 2) and a **generator**
(step 4), with the output of one call feeding the input of the next.

> **Prerequisite:** `OPENAI_API_KEY` in your `.env`. Scraping helpers come from
> [scraper.py](scraper.py).

## 1. Setup

In [ ]:
import json
import os

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

from scraper import fetch_website_content, fetch_website_links

In [ ]:
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-5.4-mini"

## 2. Step one — let the model choose the relevant links

A landing page links to dozens of places: nav, footer, legal, social. Most are
useless for a brochure. Rather than hand-writing filter rules, we hand the whole
list to the model and let it decide.

- `get_links_user_prompt` builds the instruction, embedding the **link list**
  (from `fetch_website_links(..., extract=True)`).
- `select_relevant_links` makes the call. `response_format={"type": "json_object"}`
  forces valid JSON back, which we parse into a plain Python list.

In [ ]:
def get_links_user_prompt(url: str, links: list[str]) -> str:
    return f"""Here's a list of links found on {url}. Decide which are relevant
for designing a brochure about the website — keep things like About, Products,
Careers, or news sections, and drop Terms, Privacy, login, ad-choices, and
social-share links.

Respond with a JSON object of exactly this shape, using absolute URLs:
{{"relevant_links": ["https://full.url/one", "https://full.url/two"]}}

Here are the links:
{links}"""

In [ ]:
def select_relevant_links(url: str, links: list[str]) -> list[str]:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "You select the relevant links for designing a brochure about a website. Respond in JSON.",
            },
            {"role": "user", "content": get_links_user_prompt(url=url, links=links)},
        ],
        response_format={"type": "json_object"},
    )
    data = json.loads(response.choices[0].message.content)  # type: ignore
    return data.get("relevant_links", [])

Quick check — fetch CNN's links and see what the model keeps.

> `extract=True` returns the **list of links** (a few KB). `extract=False` would
> return the raw HTML (~1M tokens for CNN) and blow past the model's per-minute
> token limit — that's the 429 trap to avoid here.

In [ ]:
url = "https://www.cnn.com"
select_relevant_links(url=url, links=fetch_website_links(url=url, extract=True))

## 3. Gather the page content and relevant links

`fetch_page_and_relevant_links` bundles step one with a scrape of the landing
page, returning everything the brochure step needs in a single dict.

In [ ]:
def fetch_page_and_relevant_links(url: str) -> dict:
    content = fetch_website_content(url=url)
    links = fetch_website_links(url=url, extract=True)   # list of links, not raw HTML
    relevant_links = select_relevant_links(url=url, links=links)
    return {"content": content, "relevant_links": relevant_links}

## 4. Step two — generate the brochure

The second LLM call. The **system prompt** defines the brochure structure; the
**user prompt** carries the data we gathered in step 3.

In [ ]:
brochure_system_prompt = """You are a helpful assistant that designs brochures for websites. You will be given the content of a website and a list of relevant links. Use this information to design a brochure for the website. The brochure should include the following sections:
1. Overview: A brief summary of the website and its purpose.
2. Key Features: A list of the key features of the website, with a brief description.
3. Target Audience: A description of the target audience for the website.
4. Relevant Links: A list of the relevant links, with a brief description of each.
Make the brochure visually appealing and easy to read. Use headings, bullet points, and other markdown formatting."""

In [ ]:
def create_brochure(url: str) -> str:
    page = fetch_page_and_relevant_links(url=url)

    # Trim the page text so a content-heavy site stays under the per-minute
    # token limit (the brochure quality barely changes past the first ~20k chars).
    content = page["content"][:20_000]

    user_prompt = f"""Here's the content of the website:
{content}

And here are some relevant links:
{page['relevant_links']}

Please design a brochure for the website based on this information."""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return response.choices[0].message.content  # type: ignore

Run the whole pipeline end-to-end and render the brochure as markdown.

In [ ]:
brochure = create_brochure(url="https://www.cnn.com")
display(Markdown(brochure))